In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../Data/FixTimeBars/EURUSD_H1_Demo_READY.csv",
                 index_col="time", parse_dates=True)

print(df.shape)
df.head()

(8323, 9)


,open,high,low,close,tick_volume,spread,real_volume,low_time,high_time
time,,,,,,,,,
2025-05-16 04:00:00,1.12044,1.12081,1.11978,1.12028,2988,8,0,2025-05-16 04:30:00,2025-05-16 04:45:00
2025-05-16 05:00:00,1.12031,1.12081,1.11988,1.12016,2412,8,0,2025-05-16 05:35:00,2025-05-16 05:15:00
2025-05-16 06:00:00,1.12016,1.12052,1.11983,1.11999,2205,8,0,2025-05-16 06:10:00,2025-05-16 06:35:00
2025-05-16 07:00:00,1.11999,1.12038,1.11958,1.12026,1787,8,0,2025-05-16 07:10:00,2025-05-16 07:55:00
2025-05-16 08:00:00,1.12026,1.12181,1.12006,1.12151,2464,8,0,2025-05-16 08:00:00,2025-05-16 08:55:00


In [2]:
df["log_return"] = np.log(df["close"]).diff()
df["future_log_return"] = df["log_return"].shift(-1)

df[["close", "log_return", "future_log_return"]].head()

,close,log_return,future_log_return
time,,,
2025-05-16 04:00:00,1.12028,NaN,-0.000107
2025-05-16 05:00:00,1.12016,-0.000107,-0.000152
2025-05-16 06:00:00,1.11999,-0.000152,0.000241
2025-05-16 07:00:00,1.12026,0.000241,0.001115
2025-05-16 08:00:00,1.12151,0.001115,-0.000294


In [3]:
import ta

# Feature 1: RSI
df["rsi_14"] = ta.momentum.RSIIndicator(close=df["close"], window=14).rsi()
df["ema_50"] = ta.trend.EMAIndicator(close=df["close"], window=50).ema_indicator()


# Feature 2: distance from EMA, as a percentage

df["ema_dist"] = (df["close"] - df["ema_50"]) / df["ema_50"]

df[["close", "rsi_14", "ema_dist", "future_log_return"]].head(60)

,close,rsi_14,ema_dist,future_log_return
time,,,,
2025-05-16 04:00:00,1.12028,NaN,NaN,-0.000107
2025-05-16 05:00:00,1.12016,NaN,NaN,-0.000152
2025-05-16 06:00:00,1.11999,NaN,NaN,0.000241
2025-05-16 07:00:00,1.12026,NaN,NaN,0.001115
2025-05-16 08:00:00,1.12151,NaN,NaN,-0.000294
2025-05-16 09:00:00,1.12118,NaN,NaN,-0.000321
2025-05-16 10:00:00,1.12082,NaN,NaN,-0.000857
2025-05-16 11:00:00,1.11986,NaN,NaN,-0.000411
2025-05-16 12:00:00,1.11940,NaN,NaN,0.000447


In [4]:
from scipy.stats import spearmanr

features = ["rsi_14", "ema_dist"]
clean = df.dropna(subset=features + ["future_log_return"])

for f in features:
    rho, pval = spearmanr(clean[f], clean["future_log_return"])
    print(f"{f:12s}  rho = {rho:+.4f}   p = {pval:.4f}")

rsi_14        rho = -0.0279   p = 0.0112
ema_dist      rho = -0.0214   p = 0.0511


In [5]:
df[features].corr(method="spearman")

,rsi_14,ema_dist
rsi_14,1.000000,0.933867
ema_dist,0.933867,1.000000


In [6]:
# Volatility: Parkinson estimator (uses high/low range)
df["parkinson"] = np.sqrt(
    (1 / (4 * np.log(2))) * (np.log(df["high"] / df["low"]) ** 2)
).rolling(20).mean()

# Time of day: hour of the bar
df["hour"] = df.index.hour

features = ["rsi_14", "ema_dist", "parkinson", "hour"]
clean = df.dropna(subset=features + ["future_log_return"])

for f in features:
    rho, pval = spearmanr(clean[f], clean["future_log_return"])
    print(f"{f:12s}  rho = {rho:+.4f}   p = {pval:.4f}")

print()
print(df[features].corr(method="spearman").round(3))

rsi_14        rho = -0.0279   p = 0.0112
ema_dist      rho = -0.0214   p = 0.0511
parkinson     rho = +0.0071   p = 0.5166
hour          rho = -0.0236   p = 0.0317

           rsi_14  ema_dist  parkinson   hour
rsi_14      1.000     0.934      0.079 -0.004
ema_dist    0.934     1.000      0.077 -0.000
parkinson   0.079     0.077      1.000 -0.019
hour       -0.004    -0.000     -0.019  1.000


In [7]:
df = df.drop(columns=["ema_dist", "parkinson"])
print(df.columns.tolist())

['open', 'high', 'low', 'close', 'tick_volume', 'spread', 'real_volume', 'low_time', 'high_time', 'log_return', 'future_log_return', 'rsi_14', 'ema_50', 'hour']


In [8]:
bucket = clean.groupby("hour")["future_log_return"].agg(["mean", "count"])
print(bucket)

              mean  count
hour                     
0     1.046853e-04    344
1     5.074515e-05    344
2     2.983890e-05    344
3    -1.121281e-04    345
4    -3.693325e-06    344
5    -1.107124e-05    344
6    -5.300093e-07    344
7     4.603103e-07    344
8     3.016114e-05    344
9    -2.463113e-05    344
10   -2.104442e-05    345
11    7.414859e-06    345
12   -4.772297e-05    346
13   -7.415912e-05    346
14    1.411966e-04    346
15    8.136333e-05    346
16    4.623483e-05    346
17    2.580466e-06    346
18   -2.876705e-05    346
19   -3.656356e-05    346
20   -2.679578e-05    345
21   -6.907881e-06    344
22   -2.271852e-05    344
23   -1.521895e-05    341


In [9]:
df = df.drop(columns=["hour"])
print(df.columns.tolist())

['open', 'high', 'low', 'close', 'tick_volume', 'spread', 'real_volume', 'low_time', 'high_time', 'log_return', 'future_log_return', 'rsi_14', 'ema_50']
